In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
dataset_path = os.path.join(BASE_DIR, "hand_gesture_dataset_processed")

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=15, brightness_range=[0.8, 1.2])
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=True
)

Found 534 images belonging to 8 classes.


In [ ]:
val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

Found 153 images belonging to 8 classes.
Found 77 images belonging to 8 classes.


In [ ]:
num_classes = len(train_data.class_indices)

In [ ]:
with open(os.path.join(BASE_DIR, "class_indices.json"), "w") as f:
    json.dump(train_data.class_indices, f)

In [ ]:
model = models.Sequential([
    # 1. Input Layer + First Convolution
    # Using BatchNormalization after pool for stable training
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),

    # 2. Second Convolution (extracts more complex features)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),

    # 3. Third Convolution
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),

    # 4. Flattening (converts 2D features into a 1D vector for the classifier)
    layers.Flatten(),

    # 5. Dense Layers (The Classifier)
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    # 6. Output Layer
    # Use 'softmax' for multi-class classification
    layers.Dense(num_classes, activation='softmax')
])

c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 24, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 11, 11, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 9, 9, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5184)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       331,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,744 (1.48 MB)

 Trainable params: 388,424 (1.48 MB)

 Non-trainable params: 320 (1.25 KB)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint_path = os.path.join(BASE_DIR, "hand_gesture_model_best.keras")
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, earlystop]

In [ ]:
print("\nTraining the model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Training the model...
Epoch 1/20
16/17 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.3865 - loss: 2.1425
Epoch 1: val_loss improved from None to 2.03716, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 1: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.5225 - loss: 1.5528 - val_accuracy: 0.1438 - val_loss: 2.0372
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.7952 - loss: 0.6162
Epoch 2: val_loss improved from 2.03716 to 2.01026, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 2: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.8071 - loss: 0.5568 - val_accuracy: 0.1307 - val_loss: 2.0103
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9240 - loss: 0.2710
Epoch 3: val_lo

In [ ]:
print("\nEvaluating on test data...")
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating on test data...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1818 - loss: 2.0004
Test Loss: 2.0004
Test Accuracy: 0.1818


In [ ]:
y_true = test_data.classes
preds = model.predict(test_data, verbose=1)
y_pred = preds.argmax(axis=1)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=[k for k, v in sorted(train_data.class_indices.items(), key=lambda x: x[1])])
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)
with open(os.path.join(BASE_DIR, "classification_report.txt"), "w") as f:
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(cr)


Confusion Matrix:
 [[ 0  0  7  0  0  0  0  0]
 [ 0  4  6  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]]

Classification Report:
                       precision    recall  f1-score   support

   background_images       0.00      0.00      0.00         7
    left_hand_images       1.00      0.40      0.57        10
level_1_speed_images       0.14      1.00      0.24        10
level_2_speed_images       0.00      0.00      0.00        10
level_3_speed_images       0.00      0.00      0.00        10
level_4_speed_images       0.00      0.00      0.00        10
   right_hand_images       0.00      0.00      0.00        10
         stop_images       0.00      0.00      0.00        10

            accuracy                           0.18        77
           macro avg       0.14      0.17      0.10        77
        weighted avg       0.15      0.18      0.11   

c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [ ]:
class_names = [k for k, v in sorted(train_data.class_indices.items(), key=lambda x: x[1])]
tick_marks = np.arange(len(class_names))

In [ ]:
# Raw confusion matrix heatmap
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm.max() / 2.0 if cm.size else 0
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, f"{cm[i, j]:d}", ha='center',
            color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix.png'))
plt.close()

# Normalized confusion matrix (rows sum to 1)
cm_norm = cm.astype('float')
row_sums = cm_norm.sum(axis=1)[:, np.newaxis]
with np.errstate(divide='ignore', invalid='ignore'):
    cm_norm = np.divide(cm_norm, row_sums)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(8, 6))
plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Normalized Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm_norm.max() / 2.0 if cm_norm.size else 0
for i, j in np.ndindex(cm_norm.shape):
    plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center',
            color='white' if cm_norm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label (normalized)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix_normalized.png'))
plt.close()

# Plot training curves
plt.figure()
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.legend()
plt.title('Loss')
plt.savefig(os.path.join(BASE_DIR, 'loss_curve.png'))
plt.close()

plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.savefig(os.path.join(BASE_DIR, 'accuracy_curve.png'))
plt.close()

In [ ]:
# Save the trained model
model_save_path = os.path.join(BASE_DIR, "hand_gesture_model.keras")
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")


Model saved to c:\Users\louis\Documents\Telerobotics\hand_gesture_model.keras
